# Imports

In [153]:
import torch
import torch.nn.functional as F
import pandas as pd
from torch_geometric.data import Data, InMemoryDataset
from transformers import AutoTokenizer, AutoModel


# Globals

In [154]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Encoders

In [155]:
class SequenceEncoder:
    def __init__(self, model_name: str = "facebook/esm2_t6_8M_UR50D"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.model.eval()

    def __call__(self, sequences: list[str]) -> torch.Tensor:
        toks = self.tokenizer(
            sequences,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024,
            return_special_tokens_mask=True,
        )
        special_mask = toks.pop("special_tokens_mask")
        toks = {k: v.to(device) for k, v in toks.items()}
        special_mask = special_mask.to(device)

        with torch.no_grad():
            outputs = self.model(**toks)
            embeddings = outputs.last_hidden_state
            attn_mask = toks["attention_mask"].bool()
            spec_mask = special_mask.bool()
            keep_mask = attn_mask & (~spec_mask)
            keep_mask_f = keep_mask.unsqueeze(-1).type_as(embeddings)
            summed = (embeddings * keep_mask_f).sum(dim=1)
            counts = keep_mask.sum(dim=1).clamp(min=1).unsqueeze(-1)
            pooled = summed / counts
        return pooled


In [156]:
from torch_geometric.nn import MessagePassing
class MPNN(MessagePassing):
    """
    Placeholder for the MPNN -- will implement later.
    Currently using SAGEConv, but later will switch to a custom MPNN
    to be able to combine the benefits of both GraphSAGE and
    custom edge features.
    """
    pass

from torch_geometric.nn import SAGEConv as MPNN


# Data interface

In [157]:
class Proteins(InMemoryDataset):
    def __init__(self, root: str, transform=None, pre_transform=None):
        self.seq_encoder = SequenceEncoder()
        super().__init__(root, transform, pre_transform)
        # Reload with weights_only=False to get Data object back
        self.data, self.slices = torch.load(self.processed_paths[0], weights_only=False)

    @property
    def raw_file_names(self) -> list[str]:
        return ["nodes.csv", "edges.csv"]

    @property
    def processed_file_names(self) -> list[str]:
        return ["data.pt"]

    def download(self):
        pass

    def process(self):
        nodes_df = pd.read_csv(self.raw_paths[0])
        edges_df = pd.read_csv(self.raw_paths[1])

        edge_index = torch.tensor(
            edges_df[["src_idx", "dst_idx"]].to_numpy().T, dtype=torch.long
        )
        edge_weight = torch.tensor(edges_df["pmi_weight"].to_numpy(), dtype=torch.float)

        sequences = nodes_df["sequence"].tolist()
        batch_size = 16
        embeds = []
        for start in range(0, len(sequences), batch_size):
            batch = sequences[start : start + batch_size]
            embeds.append(self.seq_encoder(batch))
        x = torch.cat(embeds, dim=0)

        go_terms_series = nodes_df["go_terms"].fillna("")
        vocab = set()
        for terms in go_terms_series:
            if terms:
                vocab.update(t.strip() for t in terms.split(";") if t.strip())
        go_terms = sorted(vocab)
        go_index = {term: idx for idx, term in enumerate(go_terms)}

        y = torch.zeros((len(nodes_df), len(go_terms)), dtype=torch.float)
        label_mask = torch.zeros(len(nodes_df), dtype=torch.bool)
        for idx, terms in enumerate(go_terms_series):
            if not terms:
                continue
            label_mask[idx] = True
            for term in terms.split(";"):
                term = term.strip()
                if term:
                    y[idx, go_index[term]] = 1.0

        labeled_idx = torch.nonzero(label_mask).squeeze(-1)
        g = torch.Generator().manual_seed(1)
        perm = labeled_idx[torch.randperm(labeled_idx.numel(), generator=g)]
        n = perm.numel()
        n_train = int(0.8 * n)
        n_val = int(0.1 * n)
        train_mask = torch.zeros(len(nodes_df), dtype=torch.bool)
        val_mask = torch.zeros(len(nodes_df), dtype=torch.bool)
        test_mask = torch.zeros(len(nodes_df), dtype=torch.bool)
        train_mask[perm[:n_train]] = True
        val_mask[perm[n_train:n_train + n_val]] = True
        test_mask[perm[n_train + n_val:]] = True

        data = Data(
            x=x,
            edge_index=edge_index,
            edge_weight=edge_weight,
            y=y,
            label_mask=label_mask,
            train_mask=train_mask,
            val_mask=val_mask,
            test_mask=test_mask,
        )

        # store metadata on the dataset
        self.go_terms = go_terms
        self.accessions = nodes_df["accession"].tolist()

        data, slices = self.collate([data])
        torch.save((data, slices), self.processed_paths[0])

    def len(self):
        return 1

    def get(self, idx):
        if idx != 0:
            raise IndexError
        return self.data


# Model

In [158]:
class Model(torch.nn.Module):
    def __init__(self, in_ch, hidden_ch, out_ch):
        super().__init__()
        self.conv1 = MPNN(in_ch, hidden_ch)
        self.conv2 = MPNN(hidden_ch, out_ch)

    def forward(self, x, edge_index, edge_weight=None):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=0.5, training=self.training)
        h = self.conv2(h, edge_index)
        return h


# Training Loop

## Preliminaries

### Data, model, optim, loss

In [159]:
ds = Proteins(root="synthetic_data")
data = ds[0].to(device)

model = Model(data.x.size(1), 256, data.y.size(1)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = torch.nn.BCEWithLogitsLoss()


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/var/folders/k_/ppl3vwtj2pl2yznwt8yz4b6r0000gn/T/ipykernel_86191/42507742.py:92: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  return self.data


### Eval

In [160]:
def accuracy(logits, y_true, mask, threshold=0.5):
    if mask.sum() == 0:
        return 0.0
    probs = torch.sigmoid(logits[mask])
    preds = (probs >= threshold).float()
    true = y_true[mask]
    return (preds == true).float().mean().item()

## Loop

In [161]:
EPOCHS = 50
for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    logits = model(data.x, data.edge_index)
    loss = criterion(logits[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

    if epoch == 1 or epoch % 5 == 0:
        model.eval()
        with torch.no_grad():
            logits = model(data.x, data.edge_index)
            val_acc = accuracy(logits, data.y, data.val_mask)
            test_acc = accuracy(logits, data.y, data.test_mask)
        print(f"Epoch {epoch:03d} | loss {loss.item():.4f} | val acc {val_acc:.3f} | test acc {test_acc:.3f}")


Epoch 001 | loss 0.7055 | val acc 0.776 | test acc 0.756
Epoch 005 | loss 0.5637 | val acc 0.776 | test acc 0.756
Epoch 010 | loss 0.5402 | val acc 0.776 | test acc 0.756
Epoch 015 | loss 0.5386 | val acc 0.776 | test acc 0.756
Epoch 020 | loss 0.5306 | val acc 0.776 | test acc 0.756
Epoch 025 | loss 0.5320 | val acc 0.776 | test acc 0.756
Epoch 030 | loss 0.5267 | val acc 0.776 | test acc 0.756
Epoch 035 | loss 0.5216 | val acc 0.776 | test acc 0.756
Epoch 040 | loss 0.5141 | val acc 0.776 | test acc 0.756
Epoch 045 | loss 0.5120 | val acc 0.776 | test acc 0.756
Epoch 050 | loss 0.5077 | val acc 0.776 | test acc 0.756
